# Positional Encoding

Self-attention has no idea what order the words came in.

"rat chase mouse" and "mouse chase rat" contain the same three words, so to a bare
attention layer they are the *same input in a different order* - and it produces the
same answer for both, just shuffled. This notebook shows that failure first, then fixes
it by adding a position signal to the word vectors.

In [1]:
import torch
import torch.nn as nn
import math

torch.manual_seed(0)   # so the numbers below are reproducible

In [2]:
#1. vocabulary
vocab = {
    "<pad>": 0,
    "rat": 1,
    "chase": 2,
    "mouse": 3
}

sentence1 = ["rat", "chase", "mouse"]
sentence2 = ["mouse", "chase", "rat"]

def tokenize(sentence):
    return torch.tensor([vocab[word] for word in sentence])


tokens1 = tokenize(sentence1)
tokens2 = tokenize(sentence2)

print(tokens1)
print(tokens2)

tensor([1, 2, 3])
tensor([3, 2, 1])


In [3]:
# 2 Word Embedding Layer
embedding_dim = 8

embedding = nn.Embedding(
    num_embeddings = len(vocab),
    embedding_dim = embedding_dim
)

x1 = embedding(tokens1)
x2 = embedding(tokens2)

print("rat chase mouse", x1.shape)
print(x1)

print("\nmouse chase rat", x2.shape)
print(x2)

rat chase mouse torch.Size([3, 8])
tensor([[ 0.3223, -1.2633,  0.3500,  0.3081,  0.1198,  1.2377,  1.1168, -0.2473],
        [-1.3527, -1.6959,  0.5667,  0.7935,  0.5988, -1.5551, -0.3414,  1.8530],
        [ 0.7502, -0.5855, -0.1734,  0.1835,  1.3894,  1.5863,  0.9463, -0.8437]],
       grad_fn=<EmbeddingBackward0>)

mouse chase rat torch.Size([3, 8])
tensor([[ 0.7502, -0.5855, -0.1734,  0.1835,  1.3894,  1.5863,  0.9463, -0.8437],
        [-1.3527, -1.6959,  0.5667,  0.7935,  0.5988, -1.5551, -0.3414,  1.8530],
        [ 0.3223, -1.2633,  0.3500,  0.3081,  0.1198,  1.2377,  1.1168, -0.2473]],
       grad_fn=<EmbeddingBackward0>)


An embedding layer is a lookup table. `rat` is token `1`, so it gets row 1 of the table -
the same vector whether it sits at the start of the sentence or the end.

In [4]:
# "rat" is at position 0 in sentence1 and position 2 in sentence2
print(torch.equal(x1[0], x2[2]))

True


Look closely: the second matrix is the first one with the rows *and* the columns
reversed. Reversing the sentence reversed the attention map and changed nothing else.

This is **permutation equivariance**: permute the input tokens and the output is the same
set of vectors, permuted the same way. Attention treats a sentence as a *bag* of words.

`rat chase mouse` and `mouse chase rat` mean opposite things, and the layer cannot tell
them apart. We have to inject the position ourselves.

## Sinusoidal positional encoding

Each position gets a fixed vector built from sines and cosines of different frequencies:

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right)
\qquad
PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

Even dimensions get the sine, odd dimensions the cosine. The wavelength stretches
geometrically across dimensions - early dimensions flip quickly from position to
position, later ones drift slowly - so each position ends up with a distinct signature,
and the encoding extends to sequences longer than anything seen in training.

The exponent is computed in log space (`exp(2i * -log(10000)/d)`) rather than as a
direct power, which is numerically steadier for large `d`.

In [8]:
# 4 Positional encoding
def positional_encoding(max_len, d_model):

    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()

    even_dimension_indices = torch.arange(0, d_model, 2).float()
    decay_factor = -math.log(10000.0) / d_model

    div_term = torch.exp(even_dimension_indices * decay_factor)

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    return pe


pe = positional_encoding(max_len=10, d_model=embedding_dim)

print(pe.shape)
print(pe[:3])

torch.Size([10, 8])
tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
          9.9995e-01,  1.0000e-03,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
          9.9980e-01,  2.0000e-03,  1.0000e+00]])


Note these are *not* learned - no `nn.Parameter`, no gradients. It is a fixed table,
computed once, sliced to the length of whatever sentence arrives.

In [9]:
# 5 Add position to the word vectors
seq_len = len(sentence1)

x1_pe = x1 + pe[:seq_len]
x2_pe = x2 + pe[:seq_len]

# "rat" again: position 0 in sentence1, position 2 in sentence2
print(torch.equal(x1_pe[0], x2_pe[2]))

print("\nrat at position 0")
print(x1_pe[0])

print("\nrat at position 2")
print(x2_pe[2])

False

rat at position 0
tensor([ 0.3223, -0.2633,  0.3500,  1.3081,  0.1198,  2.2377,  1.1168,  0.7527],
       grad_fn=<SelectBackward0>)

rat at position 2
tensor([ 1.2316, -1.6795,  0.5487,  1.2882,  0.1398,  2.2375,  1.1188,  0.7527],
       grad_fn=<SelectBackward0>)


Same word, two different vectors now. The word identity is still in there - the
embedding was added, not replaced - with a position offset layered on top.

Addition rather than concatenation looks lossy, but with `d_model` reasonably large the
model has room to keep the two signals separable, and it costs no extra dimensions.

## Self-attention, with positions

In [10]:
out1_pe, attn1_pe = self_attention(x1_pe)
out2_pe, attn2_pe = self_attention(x2_pe)

print("Attention weights: rat chase mouse")
print(attn1_pe)

print("\nAttention weights: mouse chase rat")
print(attn2_pe)

Attention weights: rat chase mouse
tensor([[0.3736, 0.2354, 0.3910],
        [0.3571, 0.2915, 0.3514],
        [0.3856, 0.2384, 0.3760]], grad_fn=<SoftmaxBackward0>)

Attention weights: mouse chase rat
tensor([[0.3639, 0.2442, 0.3919],
        [0.3871, 0.2907, 0.3222],
        [0.4349, 0.2284, 0.3367]], grad_fn=<SoftmaxBackward0>)


In [11]:
# Is the mirror symmetry still there?
print("Still identical?")
print(torch.allclose(attn2_pe, attn1_pe.flip(0).flip(1), atol=1e-6))

print("\nLargest disagreement:")
print((attn2_pe - attn1_pe.flip(0).flip(1)).abs().max().item())

Still identical?
False

Largest disagreement:
0.04394814372062683


The symmetry is broken. The two sentences now produce genuinely different attention
patterns, which means the layer downstream can respond to word order.

The weights here are random and untrained, so *which* token attends to which is
meaningless - the point is only that the two sentences are no longer interchangeable.
The ability to distinguish them is the prerequisite; training is what turns it into
"rat is the subject."

### Summary

| | `rat chase mouse` vs `mouse chase rat` |
|---|---|
| word embeddings alone | identical vectors, order invisible |
| self-attention alone | permutation-equivariant - same output, reshuffled |
| \+ positional encoding | distinct representations, order recoverable |

Next: causal masking, and multi-head attention.